# 04 — Modelo 1: reconstrucción y reanálisis
**Paper Cañete · Royal Society Open Science**

El archivo de entrada original del Modelo 1 (`grilla_final_modelo_logistico (1).csv`, con
`presencia_sitio_total`) **no fue recuperado**. Este notebook: (1) documenta la grilla M1
disponible (celdas de **250 m**, no 50 m como el M2 — dato para Métodos), (2) demuestra su
tautología variable-etiqueta, (3) **reconstruye** la etiqueta total con los 188 sitios
unificados, y (4) muestra que las conclusiones no dependen de la versión exacta de la etiqueta.
Todo corre en <2 min (12,826 celdas).

In [ ]:
import pandas as pd, numpy as np, warnings, json
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from imblearn.over_sampling import SMOTE

m1 = pd.read_csv('../datos/grilla_modelo1_250m_rec.csv')
print(f"celdas: {len(m1):,} (250x250 m) | presencia_sitio_nuevo: {m1.presencia_sitio_nuevo.sum()} | "
      f"presencia_total_rec: {m1.presencia_total_rec.sum()}")

## 1. La tautología del Modelo 1
Para la etiqueta **verificable** (`presencia_sitio_nuevo`, tal como venía en el archivo original),
la variable `distancia_sitios_nuevos` separa las clases **perfectamente** (~148 m):

In [ ]:
print(m1.groupby('presencia_sitio_nuevo')['distancia_sitios_nuevos'].agg(['min','max']).to_string())
umbral = (m1.distancia_sitios_nuevos <= 148).astype(int)
print(f"\n'clasificador' umbral<=148m sobre presencia_sitio_nuevo: "
      f"accuracy={(umbral==m1.presencia_sitio_nuevo).mean():.6f}")
# Misma familia de fuga que el M2 (30 m): la etiqueta se definió por proximidad y la
# distancia se calculó a los mismos sitios.

## 2. Reconstrucción de `presencia_sitio_total`
Regla: celda positiva si contiene un sitio unificado (Chebyshev ≤125 m del centroide,
media celda) o si `presencia_sitio_nuevo==1`. Da 188 positivos; la matriz de confusión de la
tesis implica ~169 — la versión original usó fuentes levemente distintas, hoy irrecuperables.
Se reporta como **M1-rec**, y abajo se muestra que el régimen de resultados es idéntico.

In [ ]:
# (la reconstrucción ya viene aplicada en el CSV; aquí la regla por transparencia)
s = pd.read_csv('../datos/sitios_unificados_188.csv')
lab = np.zeros(len(m1), int)
for x0, y0 in zip(s.x, s.y):
    lab[((np.abs(m1.X-x0)<=125)&(np.abs(m1.Y-y0)<=125)).values] = 1
assert (np.maximum(lab, m1.presencia_sitio_nuevo.values) == m1.presencia_total_rec.values).all()
y = m1.presencia_total_rec.values
print(f"reconstrucción verificada: {y.sum()} positivos | prevalencia: {y.mean():.3%}")

## 3. Pipeline original del notebook de la tesis (sobre M1-rec)
SMOTE → split 80/20 → RF(100, depth=8, balanced), 4 semillas (como en `modelo 1_versionfinal`).

In [ ]:
FEAT_ORIG = ['distancia_sitios_nuevos','distancia_mincul','altitud','pendiente']
Xr, yr = SMOTE(random_state=42).fit_resample(m1[FEAT_ORIG].values, y)
f1s = []
for seed in [1,42,123,2024]:
    Xtr,Xte,ytr,yte = train_test_split(Xr,yr,test_size=0.2,stratify=yr,random_state=seed)
    mo = RandomForestClassifier(class_weight='balanced',n_estimators=100,max_depth=8,random_state=seed).fit(Xtr,ytr)
    f1s.append(f1_score(yte, mo.predict(Xte)))
print(f"F1 = {np.mean(f1s):.4f} ± {np.std(f1s):.4f}   (tesis: 0.976 — mismo régimen inflado)")

## 4. Evaluación corregida (solo terreno: altitud, pendiente)
El M1 no tiene variables hidrológicas; su versión honesta usa solo el terreno.

In [ ]:
X = m1[['altitud','pendiente']].values.astype(np.float32)
m1['block'] = (m1.X//2000).astype(int).astype(str)+'_'+(m1.Y//2000).astype(int).astype(str)
pp = np.zeros(len(y))
for tr,te in GroupKFold(5).split(X,y,m1['block'].values):
    Xs,ys = SMOTE(random_state=42,k_neighbors=min(5,int(y[tr].sum())-1)).fit_resample(X[tr],y[tr])
    pp[te] = RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1).fit(Xs,ys).predict_proba(X[te])[:,1]
pred = (pp>=.5).astype(int); prev = y.mean(); orden = np.argsort(-pp)
print(f"spatial CV: prec={precision_score(y,pred,zero_division=0):.4f} rec={recall_score(y,pred):.4f} "
      f"PR-AUC={average_precision_score(y,pp):.4f}")
print(f"precision@85 (presupuesto real del M1): {y[orden[:85]].mean():.4f} "
      f"= {y[orden[:85]].mean()/prev:.1f}x la prevalencia ({prev:.4f})")

## Para el paper
- El M1 se reporta como **diseño exploratorio inicial** con etiqueta reconstruida (M1-rec),
  documentando que el input exacto es irrecuperable y que el régimen de resultados es insensible
  a esa diferencia (F1≈0.98–1.0 contaminado en ambas versiones).
- Diferencias M1 vs M2 a declarar en Métodos: resolución (250 m vs 50 m), variables
  (sin hidrología vs con), y prevalencia (1.5% vs 0.03%) — por eso sus matrices nunca fueron comparables.
- La comparación honesta entre modelos se hace en el campo (notebook 03) y con precisión@presupuesto.